<a href="https://colab.research.google.com/github/shoukk8-afk/Architecture-of-VAE-by-PyTorch/blob/main/VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets

In [29]:
#VAEエンコーダ
class Encoder(nn.Module):
    def __init__(self, in_channels, latent_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc1 = nn.Linear(2048, latent_dim)
        self.fc2 = nn.Linear(2048, latent_dim)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = x.view(-1, 2048)
        #平均値、対数の分散を出力させる
        z_mean = self.fc1(x)
        z_log_var = self.fc2(x)
        return z_mean, z_log_var

In [28]:
#デコーダ
class Decoder(nn.Module):
    def __init__(self, out_channels, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 2048)
        self.conv1 = nn.ConvTranspose2d(128, 128, 4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(128)
        self.conv2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, out_channels, 3, padding=1)

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 128, 4, 4)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = torch.sigmoid(self.conv4(x))
        return x

In [20]:
#サンプリング
def sampling(inputs):
    z_mean, z_log_var = inputs
    epsilon = torch.randn_like(z_mean)
    return z_mean + torch.exp(0.5 * z_log_var) * epsilon

In [21]:
#KLダイバージェンス
def kl_loss(z_mean, z_log_var):
    kl_per_sample = -0.5 * torch.sum(1 + z_log_var - z_mean.pow(2) - torch.exp(z_log_var), dim=1)
    return torch.mean(kl_per_sample)

In [22]:
#デバイスの設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [27]:
#訓練ループ
def training_loop(n_epochs, optimizer, encoder, decoder, train_loader, test_loader):
    for epoch in range(1, n_epochs + 1):
        total = 0
        loss_train = 0
        encoder.train()
        decoder.train()
        for imgs, _ in train_loader:
            #imgsをGPUに移す
            imgs = imgs.to(device=device)

            #エンコーダの適用
            z_mean, z_log_var = encoder(imgs)
            loss_kl = kl_loss(z_mean, z_log_var)

            #サンプリング
            latent = sampling((z_mean, z_log_var))

            #デコーダの適用
            decoded_imgs = decoder(latent)
            loss = F.mse_loss(decoded_imgs, imgs, reduction='sum') / imgs.shape[0]

            total_loss = loss_kl + loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            #1回のループの損失を加算して訓練データの1エボックの損失の合計を出す
            loss_train += loss.item()

        print(f"Epoch: {epoch}, Loss: {loss_train / len(train_loader)}")
